In [19]:
# Importo las librerías para conectarme y trabajar con dataframes
from sqlalchemy import create_engine
import pandas as pd
import pyodbc
from urllib.parse import quote_plus
import os
from dotenv import load_dotenv

# Cargo las variables de conexión desde el archivo .env (override por si el kernel ya tenía valores cacheados)
load_dotenv(override=True)

True

In [15]:
# Configuro la conexión a PostgreSQL de origen usando las variables del .env
usuario = os.getenv("PG_USUARIO")
password = os.getenv("PG_PASSWORD")
servidor = os.getenv("PG_SERVIDOR")
puerto = os.getenv("PG_PUERTO")
base_datos = os.getenv("PG_BASE_DATOS")

cadena = (
    f"postgresql+psycopg2://"
    f"{usuario}:{password}@"
    f"{servidor}:{puerto}/"
    f"{base_datos}"
)

engine_origen = create_engine(cadena)

try:
    with engine_origen.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [16]:
# Extraigo los datos de clientes desde la base de datos origen
consulta_cliente = "SELECT * FROM tienda.cliente"
df_cliente = pd.read_sql_query(consulta_cliente, engine_origen)

In [20]:
# Configuro la conexión al stage en SQL Server usando las variables del .env
servidor_sqlserver = os.getenv("SQLSERVER_SERVIDOR")
usuario_sqlserver = os.getenv("SQLSERVER_USUARIO")
password_sqlserver = os.getenv("SQLSERVER_PASSWORD")
base_datos_stage = os.getenv("SQLSERVER_BASE_DATOS")
esquema_stage = os.getenv("SQLSERVER_ESQUEMA")
driver_sqlserver = os.getenv("SQLSERVER_DRIVER")

cadena_sqlserver = (
    f"mssql+pyodbc://{usuario_sqlserver}:{quote_plus(password_sqlserver)}@"
    f"{servidor_sqlserver}/{base_datos_stage}?driver={quote_plus(driver_sqlserver)}"
)

engine_stage = create_engine(cadena_sqlserver)
try:
    with engine_stage.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [21]:
# Agrego la fecha y hora de carga al dataframe de clientes
df_cliente["fecha_carga"] = pd.Timestamp.now()

In [22]:
# Reviso una muestra de clientes antes de cargarla
df_cliente.head()

,cliente_id,nombres,apellidos,email,telefono,fecha_nacimiento,genero,direccion,ciudad,departamento,pais,fecha_registro,activo,fecha_creacion,fecha_modificacion,fecha_carga
0,1,Marco,Flores Cabrera,None,+591 78592709,1986-08-25,Masculino,Calle Sánchez #741,Santa Cruz,Santa Cruz,Bolivia,2026-06-29,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:10:44.434305
1,2,Gabriel,Sánchez Sánchez,cliente0002@correo.bo,+591 73504069,1979-10-26,Femenino,Calle Mamani #783,Santa Cruz,Santa Cruz,Bolivia,2024-08-12,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:10:44.434305
2,3,Gabriel,Ramírez Silva,cliente0003@correo.bo,+591 79868482,1973-12-10,Femenino,Calle Silva #233,Sucre,Chuquisaca,Bolivia,2023-09-01,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:10:44.434305
3,4,Daniel,Flores Ramírez,cliente0004@correo.bo,+591 76178679,1978-09-26,Femenino,Calle Romero #857,El Alto,La Paz,Bolivia,2025-06-22,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:10:44.434305
4,5,Elena,Rojas Alarcón,cliente0005@correo.bo,+591 76843959,1971-07-20,Masculino,Calle Condori #66,Potosí,Potosí,Bolivia,2024-07-05,False,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:10:44.434305


In [23]:
# Cargo los clientes en la tabla del stage
df_cliente.to_sql(
    name="cliente",
    con=engine_stage,
    schema=esquema_stage,
    if_exists="append",
    index=False
)

14

In [24]:
# Verifico el contenido de clientes en el stage
consulta_stage = f"SELECT * FROM {esquema_stage}.cliente"
df_cliente_stage = pd.read_sql_query(consulta_stage, engine_stage)
df_cliente_stage.head()

,cliente_id,nombres,apellidos,email,telefono,fecha_nacimiento,genero,direccion,ciudad,departamento,pais,fecha_registro,activo,fecha_creacion,fecha_modificacion,fecha_carga
0,1,Marco,Flores Cabrera,None,+591 78592709,1986-08-25,Masculino,Calle Sánchez #741,Santa Cruz,Santa Cruz,Bolivia,2026-06-29,True,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:10:44.433
1,2,Gabriel,Sánchez Sánchez,cliente0002@correo.bo,+591 73504069,1979-10-26,Femenino,Calle Mamani #783,Santa Cruz,Santa Cruz,Bolivia,2024-08-12,True,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:10:44.433
2,3,Gabriel,Ramírez Silva,cliente0003@correo.bo,+591 79868482,1973-12-10,Femenino,Calle Silva #233,Sucre,Chuquisaca,Bolivia,2023-09-01,True,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:10:44.433
3,4,Daniel,Flores Ramírez,cliente0004@correo.bo,+591 76178679,1978-09-26,Femenino,Calle Romero #857,El Alto,La Paz,Bolivia,2025-06-22,True,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:10:44.433
4,5,Elena,Rojas Alarcón,cliente0005@correo.bo,+591 76843959,1971-07-20,Masculino,Calle Condori #66,Potosí,Potosí,Bolivia,2024-07-05,False,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:10:44.433
